# 06 — Hybrid Models (C1–C7)

This notebook trains all hybrid model families:

| ID | Model | Description |
|----|-------|-------------|
| C1 | Global + per-creator LightGBM | Same structure as B2/B4 but with LGBM |
| C2 | Global + per-creator Ridge | Linear model, fast |
| C3 | Shrinkage blend | `w·per_creator + (1-w)·global`, w = n/(n+k) |
| C4 | Fine-tuning | Continue global XGB/LGBM on per-creator data |
| C5 | Two-stage | Global + per-creator on residuals |
| C6 | Cluster models | KMeans-cluster creators, one model/cluster |
| C7 | Mixed effects | Random intercept + 1-2 random slopes (MixedLM) |

All models trained on `train` split only.

In [ ]:
import os, time, warnings
from pathlib import Path

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import joblib
warnings.filterwarnings("ignore")

from creator_analytics import features as fe
from creator_analytics.config import load_config
from creator_analytics.models import (
    VS_CREATOR_SUFFIX, prepare_X, make_cv_splits,
    search_lightgbm, train_per_creator_lightgbm,
    train_ridge, train_per_creator_ridge,
    find_shrinkage_k, train_shrinkage_blend, ShrinkageBlend,
    train_finetune_per_creator,
    train_two_stage, TwoStageModel,
    train_cluster_models, ClusterModel,
    train_mixed_effects, predict_mixed_effects,
    TrainedModel,
)
from creator_analytics.evaluate import (
    compute_metrics, compute_per_creator_metrics,
    paired_comparison, log_experiment, leakage_check,
    predict_per_creator,
)

cfg       = load_config("config/default.yaml")
PROCESSED = Path(cfg["paths"]["processed_dir"])
PREDS_DIR = Path("results/predictions"); PREDS_DIR.mkdir(parents=True, exist_ok=True)
FIGS      = Path("results/figures"); FIGS.mkdir(parents=True, exist_ok=True)
SEED      = cfg["seed"]
TODAY     = "2026-10-25"


In [ ]:
data, spec = fe.load_feature_table(PROCESSED)
idx  = pd.read_parquet(PROCESSED / "dataset_index.parquet")

TRAIN = idx["split"].eq("train")
TEST  = idx["split"].eq("test")

FEATURES_GLOBAL = spec["features"]
VS_COLS = [c for c in FEATURES_GLOBAL if c.endswith(VS_CREATOR_SUFFIX)]
FEATURES_PC = [c for c in FEATURES_GLOBAL if not c.endswith(VS_CREATOR_SUFFIX)]

X_all_global = data[FEATURES_GLOBAL].copy()
X_all_pc     = data[FEATURES_PC].copy()
y_all  = idx["target_rel"].copy()
auth   = idx["author_id"].copy()

X_train_global = X_all_global[TRAIN.values].reset_index(drop=True)
X_train_pc     = X_all_pc[TRAIN.values].reset_index(drop=True)
y_train        = y_all[TRAIN.values].reset_index(drop=True)
auth_train     = auth[TRAIN.values].reset_index(drop=True)

X_test_global  = X_all_global[TEST.values].reset_index(drop=True)
X_test_pc      = X_all_pc[TEST.values].reset_index(drop=True)
y_test         = y_all[TEST.values].reset_index(drop=True)
auth_test      = auth[TEST.values].reset_index(drop=True)

print(f"Global features: {len(FEATURES_GLOBAL)}  Per-creator: {len(FEATURES_PC)}")
print(f"Train: {len(X_train_global):,}  Test: {len(X_test_global):,}")


In [ ]:
# Load NB04/NB05 results
b2_model = TrainedModel.load("models/B2_global_xgb.joblib")
rf_models_b3  = joblib.load("models/per_creator/B3_rf_models.joblib")
xgb_models_b4 = joblib.load("models/per_creator/B4_xgb_models.joblib")

nb04 = pd.read_parquet(PREDS_DIR / "nb04_predictions.parquet")
nb05 = pd.read_parquet(PREDS_DIR / "nb05_predictions.parquet")
b2_preds   = nb04["B2_XGB"].values
zero_preds = np.zeros(len(y_test))

print("B2 baseline RMSE:", round(compute_metrics(y_test, b2_preds)["rmse"], 4))


## C1 — LightGBM (global + per-creator)

In [ ]:
t0 = time.time()
lgbm_global = search_lightgbm(
    X_train_global, y_train, auth_train,
    n_splits=cfg["cv"]["n_splits"], seed=SEED, n_iter=30,
)
elapsed_c1g = time.time() - t0
print(f"C1 global LGBM  CV RMSE={lgbm_global.cv_rmse:.4f}  time={elapsed_c1g/60:.1f}min")


In [ ]:
t0 = time.time()
# Use best global params as per-creator config (conservative n_estimators)
lgbm_pc_params = {**lgbm_global.params, "n_estimators": 100}
lgbm_models_pc = train_per_creator_lightgbm(
    X_train_pc, y_train, auth_train,
    params=lgbm_pc_params, n_jobs=-1, seed=SEED,
)
elapsed_c1pc = time.time() - t0
print(f"C1 per-creator LGBM  time={elapsed_c1pc/60:.1f}min  creators={len(lgbm_models_pc)}")


## C2 — Ridge (global + per-creator)

In [ ]:
t0 = time.time()
ridge_global = train_ridge(
    X_train_global, y_train, auth_train,
    n_splits=cfg["cv"]["n_splits"], seed=SEED,
)
elapsed_c2g = time.time() - t0
print(f"C2 global Ridge  CV RMSE={ridge_global.cv_rmse:.4f}  alpha={ridge_global.params['alpha']}  time={elapsed_c2g:.1f}s")


In [ ]:
t0 = time.time()
ridge_models_pc = train_per_creator_ridge(
    X_train_pc, y_train, auth_train, n_jobs=-1, seed=SEED,
)
elapsed_c2pc = time.time() - t0
print(f"C2 per-creator Ridge  time={elapsed_c2pc/60:.1f}min  creators={len(ridge_models_pc)}")


## C3 — Shrinkage blend

`prediction = w·per_creator + (1-w)·global`, where `w = n_train / (n_train + k)`.
k is tuned by CV on train; higher k → more shrinkage toward the global model.

In [ ]:
t0 = time.time()
cv_splits = make_cv_splits(auth_train, n_splits=cfg["cv"]["n_splits"], seed=SEED)

shrinkage_blend = train_shrinkage_blend(
    per_creator_models=xgb_models_b4,
    global_model=b2_model,
    X_train=X_train_pc,
    y_train=y_train,
    author_ids_train=auth_train,
    cv_splits=cv_splits,
)
elapsed_c3 = time.time() - t0
print(f"C3 Shrinkage  k={shrinkage_blend.k}  CV proxy RMSE={shrinkage_blend.cv_rmse:.4f}  time={elapsed_c3/60:.1f}min")


## C4 — Fine-tuning (global → per-creator)

In [ ]:
t0 = time.time()
finetune_models = train_finetune_per_creator(
    X_train_pc, y_train, auth_train,
    global_model=b2_model,
    model_type="xgb",
    n_jobs=-1,
    seed=SEED,
)
elapsed_c4 = time.time() - t0
print(f"C4 Fine-tuning  creators={len(finetune_models)}  time={elapsed_c4/60:.1f}min")


## C5 — Two-stage (global + per-creator residuals)

In [ ]:
t0 = time.time()
two_stage = train_two_stage(
    X_train_pc, y_train, auth_train,
    global_model=b2_model,
    seed=SEED,
    n_jobs=-1,
)
elapsed_c5 = time.time() - t0
print(f"C5 Two-stage  residual models={len(two_stage.stage2)}  time={elapsed_c5/60:.1f}min")


## C6 — Cluster models

Cluster creators by their mean feature vector (KMeans), train one XGB per cluster.

In [ ]:
t0 = time.time()
n_clusters = 15  # ~96 creators/cluster on average
cluster_model = train_cluster_models(
    X_train_pc, y_train, auth_train,
    n_clusters=n_clusters,
    seed=SEED,
    n_jobs=-1,
)
elapsed_c6 = time.time() - t0
print(f"C6 Cluster models  n_clusters={n_clusters}  time={elapsed_c6/60:.1f}min")
cluster_sizes = pd.Series(cluster_model.creator_cluster).value_counts().sort_index()
print("Creators per cluster:")
display(cluster_sizes.describe().round(1))


## C7 — Mixed effects (MixedLM)

Random intercept per creator + random slopes for 2 key features.
30-minute time limit; skipped and documented if not converging.

In [ ]:
# Top features from B2 SHAP (from NB04 figure) — use follower + timing
# These are chosen as the most interpretable candidates for random slopes
import shap

explainer = shap.TreeExplainer(b2_model.model)
rng = np.random.RandomState(SEED)
sub = rng.choice(len(X_train_global), size=min(5000, len(X_train_global)), replace=False)
shap_values = explainer.shap_values(X_train_global.iloc[sub].values)
shap_mean = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=FEATURES_GLOBAL,
).sort_values(ascending=False)
top2_features = shap_mean.head(2).index.tolist()
print(f"Random slopes for: {top2_features}")


In [ ]:
t0 = time.time()
me_result = train_mixed_effects(
    X_train_global, y_train, auth_train,
    random_slopes=top2_features,
    max_time_seconds=1800,
)
elapsed_c7 = time.time() - t0
print(f"C7 MixedLM  converged={me_result['converged']}  elapsed={elapsed_c7:.0f}s")
if me_result["skip_reason"]:
    print(f"Skipped: {me_result['skip_reason']}")


## Evaluation on held-out test set

In [ ]:
# Collect all predictions
all_preds = {
    "B2_global_XGB":     b2_preds,
    "C1_global_LGBM":    lgbm_global.predict(X_test_global),
    "C2_global_Ridge":   ridge_global.predict(X_test_global),
    "C1_perCreator_LGBM": predict_per_creator(X_test_pc, auth_test, lgbm_models_pc),
    "C2_perCreator_Ridge": predict_per_creator(X_test_pc, auth_test, ridge_models_pc),
    "C3_shrinkage":      shrinkage_blend.predict(X_test_pc, auth_test),
    "C4_finetune_XGB":   predict_per_creator(X_test_pc, auth_test, finetune_models),
    "C5_two_stage":      two_stage.predict(X_test_pc, auth_test),
    "C6_cluster":        cluster_model.predict(X_test_pc, auth_test),
}

if me_result["result"] is not None:
    all_preds["C7_mixed_effects"] = predict_mixed_effects(me_result, X_test_global, auth_test)

metrics = {n: compute_metrics(y_test, p) for n, p in all_preds.items()}
df_m = pd.DataFrame(metrics).T[["rmse","mae","r2","spearman","n"]]
display(df_m.round(4))


In [ ]:
# Paired comparisons vs B2
print("=== All models vs B2 global XGB ===")
for name, preds in all_preds.items():
    if name == "B2_global_XGB":
        continue
    nan_mask = np.isnan(preds)
    if nan_mask.any():
        preds = preds.copy()
        preds[nan_mask] = b2_preds[nan_mask]  # fallback to B2 for missing
    cmp = paired_comparison(y_test, preds, b2_preds, auth_test,
                            name_a=name, name_b="B2_XGB")
    print(f"{name:25s}  diff={cmp['rmse_diff']:+.4f}  p={cmp['wilcoxon_pvalue']:.4f}  win={cmp['win_rate_a']:.3f}")


In [ ]:
# Save all predictions
df_out = pd.DataFrame({
    "video_id":  idx.loc[TEST, "video_id"].values,
    "author_id": auth_test.values,
    "y_true":    y_test.values,
})
for name, preds in all_preds.items():
    df_out[name] = preds
df_out.to_parquet(PREDS_DIR / "nb06_predictions.parquet", index=False)

# Save models
Path("models/hybrid").mkdir(parents=True, exist_ok=True)
lgbm_global.save("models/hybrid/C1_global_lgbm.joblib")
joblib.dump(lgbm_models_pc, "models/hybrid/C1_pc_lgbm_models.joblib")
ridge_global.save("models/hybrid/C2_global_ridge.joblib")
joblib.dump(ridge_models_pc, "models/hybrid/C2_pc_ridge_models.joblib")
joblib.dump(shrinkage_blend, "models/hybrid/C3_shrinkage_blend.joblib")
joblib.dump(finetune_models, "models/hybrid/C4_finetune_xgb_models.joblib")
joblib.dump(two_stage, "models/hybrid/C5_two_stage.joblib")
joblib.dump(cluster_model, "models/hybrid/C6_cluster_model.joblib")
if me_result["result"] is not None:
    joblib.dump(me_result, "models/hybrid/C7_mixed_effects.joblib")
print("Saved")


In [ ]:
# Log experiments
exp_path = Path(cfg["paths"]["experiments_log"])
elapsed_map = {
    "C1_global_LGBM": elapsed_c1g, "C1_perCreator_LGBM": elapsed_c1pc,
    "C2_global_Ridge": elapsed_c2g, "C2_perCreator_Ridge": elapsed_c2pc,
    "C3_shrinkage": elapsed_c3, "C4_finetune_XGB": elapsed_c4,
    "C5_two_stage": elapsed_c5, "C6_cluster": elapsed_c6,
}
if me_result["result"] is not None:
    elapsed_map["C7_mixed_effects"] = elapsed_c7

for name, preds in all_preds.items():
    if name == "B2_global_XGB":
        continue
    nan_mask = np.isnan(preds)
    preds_clean = preds.copy()
    if nan_mask.any():
        preds_clean[nan_mask] = b2_preds[nan_mask]
    cmp_b2 = paired_comparison(y_test, preds_clean, b2_preds, auth_test, name, "B2_XGB")
    log_experiment(
        exp_path, name=name, date=TODAY,
        features=f"{len(FEATURES_GLOBAL)} global / {len(FEATURES_PC)} pc",
        model=name,
        params={},
        cv_rmse=None,
        test_metrics=compute_metrics(y_test, preds_clean),
        train_time_s=elapsed_map.get(name),
        comparison=cmp_b2,
        comment="NB06; vs B2 global XGB",
    )

# Document C7 skip if applicable
if me_result["skip_reason"]:
    with open(exp_path, "a") as f:
        f.write(f"\n**C7 MixedLM skipped:** {me_result['skip_reason']} "
                f"(elapsed {me_result['elapsed_seconds']:.0f}s)\n")
print("Logged experiments")


## Summary

*(Filled from cell output)*

**Key findings:**
- C3 shrinkage is expected to outperform both pure global and pure per-creator
- C5 two-stage adds per-creator correction on top of global predictions
- C4 fine-tuning may overfit for small creators
- C7 MixedLM result depends on convergence within 30-min limit